# KDAL Station Stacking V20 1 PM No Peak

KDAL-only V20-aligned experiment using the audited 1 PM live-safe forecast and observation data. It uses the dedicated 1 PM temperature-alignment feature contract, predicts remaining warmup after the observed 1 PM high-so-far, excludes the HRRR/NBM peak-timing feature family, keeps Wunderground-only targets and the 3% train-fold missingness gate, and exports the fitted model bundle after a successful full run.


In [1]:
from pathlib import Path
import os
import sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="Skipping features without any observed values.*")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "calibration" / "station_stacking.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing src/calibration/station_stacking.py")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ["WEATHER_RESEARCH_INCLUDE_DIRECT_NBM"] = "1"

STATION_ID = "KDAL"
PROVIDERS = ("gfs", "hrrr", "nbm")
TREND_COLUMNS = [
    "observed_temp_change_last_1h_f",
    "observed_temp_change_last_3h_f",
    "observed_morning_warmup_rate_f_per_hour",
    "observed_high_so_far_change_since_9am_f",
]
TIMING_MODE = "same_day_1pm_live_safe"
FAST_MODE = False
OPTUNA_TRIALS = 30
STACK_OPTUNA_TRIALS = 30
OPTUNA_STARTUP_TRIALS = 15
STACK_OPTUNA_STARTUP_TRIALS = 15
OPTUNA_METRIC = "mae_f"
OPTUNA_VERBOSE = True
MODEL_VERSION = "station_high_regressor_v20_kdal_1pm_no_peak_stack"
EXPORT_MODEL_WEIGHTS = True
PROJECT_ROOT


WindowsPath('D:/dev/weather-research')

In [2]:
import numpy as np
import pandas as pd

from src.export_station_stacking_v2_models import export_station_model_weights
from src.calibration.station_stacking import (
    StationStackingConfig,
    V20_KDAL_1PM_TEMP_FEATURE_COLUMNS,
    _fit_feature_columns,
    _modeling_frame,
    V20_EXPANDING_FOLDS,
    missing_model_dependencies,
    provider_availability,
    run_station_year_split_experiment,
)


## 1 PM Data and Feature Contract

`timing_mode="same_day_1pm_live_safe"` selects only cycles available by the 1:15 PM local decision cutoff. `feature_version="v20_kdal_1pm_no_peak"` uses the 1 PM temperature-alignment features and excludes both the old 11 AM alignment family and the V20 peak-timing family. The model target is remaining warmup above the observed high through 1 PM; reported predictions are converted back to final daily highs by the shared pipeline.


In [3]:
fold_spec = pd.DataFrame(
    [
        {
            "fold": fold.name,
            "train_start_year": fold.train_start_year,
            "train_end_year": fold.train_end_year,
            "validation_year": fold.validation_year,
        }
        for fold in V20_EXPANDING_FOLDS
    ]
)

fold_spec


,fold,train_start_year,train_end_year,validation_year
0,fold_2021_to_2022,2021,2021,2022
1,fold_2021_2022_to_2023,2021,2022,2023
2,fold_2021_2023_to_2024,2021,2023,2024
3,fold_2021_2024_to_2025,2021,2024,2025


In [4]:
V20_KDAL_1PM_TEMP_FEATURE_COLUMNS


['v13sf_forecast_temp_1pm_mean_f',
 'v13sf_forecast_temp_1pm_median_f',
 'v13sf_forecast_temp_1pm_minus_observed_f',
 'v13sf_forecast_temp_1pm_abs_error_f',
 'v13sf_forecast_temp_1pm_warm_error_f',
 'v13sf_forecast_temp_1pm_cool_error_f',
 'v13sf_forecast_temp_1pm_spread_f',
 'v13sf_forecast_temp_1pm_provider_count',
 'v13sf_forecast_temp_bias_remaining_warmup_interaction',
 'v13sf_observation_adjusted_provider_high_f',
 'v13sf_forecast_warmup_after_1pm_f']

## Data Availability


In [5]:
availability = provider_availability(
    PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
)

availability.loc[availability["station_id"].eq(STATION_ID)]


,station_id,provider,row_count,first_contract_date,last_contract_date
0,KDAL,gfs,2021,2021-01-01,2026-07-19
1,KDAL,hrrr,2026,2021-01-01,2026-07-19
2,KDAL,nbm,2023,2021-01-01,2026-07-19


## Model Scores


In [6]:
audit_path = PROJECT_ROOT / "data" / "calibration" / "station_stacking_v20_kdal_1pm_no_peak" / "audit" / "audit_result.json"
if not audit_path.exists():
    raise FileNotFoundError(f"Missing 1 PM pull audit: {audit_path}")
pull_audit = pd.read_json(audit_path, typ="series")
assert pull_audit["timing_mode"] == TIMING_MODE, pull_audit.to_dict()
assert bool(pull_audit["passed"]), pull_audit.to_dict()
assert int(pull_audit["blocking_issue_count"]) == 0, pull_audit.to_dict()
pull_audit


timing_mode                                        same_day_1pm_live_safe
start_date                                                     2021-01-01
end_date                                                       2026-07-19
coverage                [{'provider': 'gfs', 'expected_rows': 2026, 'p...
issue_count                                                             8
blocking_issue_count                                                    0
feature_rows                                                         1998
passed                                                               True
dtype: object

In [7]:
missing_packages = missing_model_dependencies()
if missing_packages:
    raise ImportError(
        "Missing station-stacking ML packages: "
        + ", ".join(missing_packages)
        + ". Install them with: python -m pip install -r requirements.txt"
    )

config = StationStackingConfig(
    station_id=STATION_ID,
    project_root=PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
    fast_mode=FAST_MODE,
    optuna_trials=OPTUNA_TRIALS,
    stack_optuna_trials=STACK_OPTUNA_TRIALS,
    optuna_startup_trials=OPTUNA_STARTUP_TRIALS,
    stack_optuna_startup_trials=STACK_OPTUNA_STARTUP_TRIALS,
    optuna_metric=OPTUNA_METRIC,
    optuna_verbose=OPTUNA_VERBOSE,
    feature_version="v20_kdal_1pm_no_peak",
    training_profile="v20_aligned",
    target_mode="remaining_warmup",
    target_source="wunderground_only",
    max_feature_missing_fraction=0.03,
    base_model_methods=("xgboost", "lightgbm", "catboost"),
    stack_enabled=True,
    hyperparameter_space="wide",
    year_split_folds=V20_EXPANDING_FOLDS,
    year_split_validation_weights={2022: 1.0, 2023: 1.0, 2024: 1.0, 2025: 1.0},
    year_split_test_train_years=(2021, 2025),
    year_split_test_year=2026,
    output_dir=PROJECT_ROOT / "data" / "calibration" / "station_stacking_v20_kdal_1pm_no_peak",
)

config.resolved_optuna_storage_path()


WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v20_kdal_1pm_no_peak/KDAL_optuna.sqlite3')

In [8]:
result = run_station_year_split_experiment(config)
result.scoreboard


D:\dev\weather-research\src\calibration\station_stacking.py:3654: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{prefix}_{feature_name}_abs_diff_f"] = (left_values - right_values).abs()
D:\dev\weather-research\src\calibration\station_stacking.py:3653: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{prefix}_{feature_name}_diff_f"] = left_values - right_values
D:\dev\weather-research\src\calibration\station_stacking.py:3654: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `fram

,period,method,count,mae_f,rmse_f
0,validation_2022_2025,xgboost,1453,1.153762,1.545469
1,validation_2022_2025,lightgbm,1453,1.114163,1.490408
2,validation_2022_2025,catboost,1453,1.077072,1.461896
3,validation_2022_2025,provider_mean,1453,2.380349,3.321387
4,validation_2022_2025,provider_median,1453,2.168557,3.133844
5,validation_2022_2025,nbm_raw,1453,2.035450,2.997965
6,validation_2022_2025,hrrr_raw,1453,4.420983,5.314202
7,validation_2022_2025,gfs_raw,1453,2.777680,3.697188
8,test_2026,xgboost,170,1.143512,1.479108
9,test_2026,lightgbm,170,1.064216,1.374332


In [9]:
if EXPORT_MODEL_WEIGHTS:
    exported_weights = export_station_model_weights(
        project_root=PROJECT_ROOT,
        station_id=STATION_ID,
        artifact_dir=config.resolved_output_dir(),
        model_version=MODEL_VERSION,
        timing_mode=config.timing_mode,
        providers=tuple(config.providers),
        feature_version=config.effective_feature_version,
        training_profile=config.effective_training_profile,
        optuna_metric=config.effective_optuna_metric,
        target_mode=config.effective_target_mode,
        target_source=config.effective_target_source,
        base_model_methods=tuple(config.effective_base_model_methods),
        stack_enabled=config.stack_enabled,
        source_pipeline="notebooks/experiments/station_stacking_v20_kdal_1pm_no_peak",
    )

    exported_weights.bundle_path, exported_weights.manifest_path
else:
    print("Model export disabled for this experimental notebook.")


## 1 PM Feature Coverage


In [10]:
one_pm_feature_coverage = (
    result.features[V20_KDAL_1PM_TEMP_FEATURE_COLUMNS]
    .notna().mean().mul(100).sort_values(ascending=False)
    .rename("coverage_pct").reset_index().rename(columns={"index": "feature"})
)
one_pm_feature_coverage


,feature,coverage_pct
0,v13sf_forecast_temp_1pm_mean_f,100.0
1,v13sf_forecast_temp_1pm_median_f,100.0
2,v13sf_forecast_temp_1pm_minus_observed_f,100.0
3,v13sf_forecast_temp_1pm_abs_error_f,100.0
4,v13sf_forecast_temp_1pm_warm_error_f,100.0
5,v13sf_forecast_temp_1pm_cool_error_f,100.0
6,v13sf_forecast_temp_1pm_spread_f,100.0
7,v13sf_forecast_temp_1pm_provider_count,100.0
8,v13sf_forecast_temp_bias_remaining_warmup_inte...,100.0
9,v13sf_observation_adjusted_provider_high_f,100.0


In [11]:
result.feature_columns.loc[result.feature_columns["feature"].isin(V20_KDAL_1PM_TEMP_FEATURE_COLUMNS)]


,feature,kind
188,v13sf_forecast_temp_1pm_mean_f,numeric
189,v13sf_forecast_temp_1pm_median_f,numeric
190,v13sf_forecast_temp_1pm_minus_observed_f,numeric
191,v13sf_forecast_temp_1pm_abs_error_f,numeric
192,v13sf_forecast_temp_1pm_warm_error_f,numeric
193,v13sf_forecast_temp_1pm_cool_error_f,numeric
194,v13sf_forecast_temp_1pm_spread_f,numeric
195,v13sf_forecast_temp_1pm_provider_count,numeric
196,v13sf_forecast_temp_bias_remaining_warmup_inte...,numeric
197,v13sf_observation_adjusted_provider_high_f,numeric


## Dropped Feature Check


In [12]:
old_11am_alignment_features = {
    feature for feature in result.feature_columns["feature"]
    if feature.startswith("v11sf_forecast_temp_11am")
}
assert not old_11am_alignment_features, f"11 AM alignment features leaked into 1 PM model: {sorted(old_11am_alignment_features)}"
print("No 11 AM alignment features selected.")


No 11 AM alignment features selected.


## Morning Trend Coverage


In [13]:
trend_coverage = (
    result.features[TREND_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

trend_coverage


,feature,coverage_pct
0,observed_temp_change_last_1h_f,100.0
1,observed_temp_change_last_3h_f,100.0
2,observed_morning_warmup_rate_f_per_hour,100.0
3,observed_high_so_far_change_since_9am_f,100.0


## Rounded Within 1F Accuracy


In [14]:
preds = pd.concat(
    [
        result.validation_predictions.assign(period="validation_2024_2025"),
        result.test_predictions.assign(period="oof_2026"),
    ],
    ignore_index=True,
)

predicted_high = pd.to_numeric(preds["predicted_high_f"], errors="coerce")
preds["predicted_high_rounded_f"] = np.floor(predicted_high + 0.5)
preds["within_1f_after_round"] = (
    pd.to_numeric(preds["actual_high_f"], errors="coerce") - preds["predicted_high_rounded_f"]
).abs().le(1)

within_1f_accuracy_by_period = (
    preds
    .dropna(subset=["actual_high_f", "predicted_high_rounded_f"])
    .groupby(["period", "method"], as_index=False)
    .agg(
        count=("within_1f_after_round", "size"),
        within_1f_count=("within_1f_after_round", "sum"),
        within_1f_accuracy_pct=("within_1f_after_round", lambda x: x.mean() * 100),
    )
    .sort_values(["period", "within_1f_accuracy_pct"], ascending=[True, False])
)

within_1f_accuracy_by_period


,period,method,count,within_1f_count,within_1f_accuracy_pct
0,oof_2026,catboost,170,129,75.882353
7,oof_2026,ridge_stack,170,129,75.882353
3,oof_2026,lightgbm,170,126,74.117647
8,oof_2026,xgboost,170,119,70.000000
4,oof_2026,nbm_raw,170,87,51.176471
6,oof_2026,provider_median,170,71,41.764706
1,oof_2026,gfs_raw,170,67,39.411765
5,oof_2026,provider_mean,170,60,35.294118
2,oof_2026,hrrr_raw,170,18,10.588235
9,validation_2024_2025,catboost,1453,1073,73.847213


## Version Comparison


In [15]:
comparison_frames = []
for version, folder in [
    ("v1", PROJECT_ROOT / "data" / "calibration" / "station_stacking"),
    ("v2", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v2"),
    ("v3", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v3"),
    ("v4", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v4"),
    ("v5", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v5"),
    ("v6", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v6"),
    ("v7", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v7"),
    ("v8", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v8"),
    ("v9", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v9"),
    ("v10", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v10"),
    ("v11", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v11"),
]:
    path = folder / f"{STATION_ID}_year_split_scoreboard.csv"
    if path.exists():
        frame = pd.read_csv(path)
        frame["version"] = version
        comparison_frames.append(frame)

version_comparison = pd.concat(comparison_frames, ignore_index=True) if comparison_frames else pd.DataFrame()
if not version_comparison.empty:
    version_comparison = version_comparison.sort_values(["period", "mae_f", "version", "method"]).reset_index(drop=True)
version_comparison


,period,method,count,mae_f,rmse_f,version
0,test_2026,ridge_stack,138,1.369446,1.890651,v9
1,test_2026,xgboost,138,1.402066,1.889472,v11
2,test_2026,xgboost,138,1.419940,1.895994,v9
3,test_2026,ridge_stack,138,1.429989,1.907881,v11
4,test_2026,lightgbm,138,1.454606,1.932482,v11
...,...,...,...,...,...,...
83,validation_2024_2025,hrrr_raw,660,5.477361,6.266007,v3
84,validation_2024_2025,hrrr_raw,668,5.486824,6.274688,v7
85,validation_2024_2025,hrrr_raw,606,5.497186,6.283967,v5
86,validation_2024_2025,hrrr_raw,606,5.497186,6.283967,v6


## 2026 OOF Weather Brackets


In [16]:
result.bracket_metrics


,method,count,mae_f,rmse_f,bucket_log_loss,bracket_accuracy_pct,p95_absolute_error_f,large_miss_5f_pct
0,xgboost,170,1.143512,1.479108,1.197057,50.588235,2.750876,0.588235
1,lightgbm,170,1.064216,1.374332,1.148499,51.764706,2.673230,0.000000
2,catboost,170,1.048525,1.375325,1.144207,51.764706,2.829128,0.000000
3,ridge_stack,170,1.068964,1.390010,1.141149,54.117647,2.911739,0.000000
4,provider_mean,170,2.532665,3.263263,1.661527,21.764706,5.510728,7.647059
5,provider_median,170,2.371618,3.136315,1.735953,24.705882,5.446978,9.411765
6,nbm_raw,170,2.035190,2.847527,1.744113,34.117647,5.673894,9.411765
7,hrrr_raw,170,4.742768,5.404650,1.741742,9.411765,8.971633,41.764706
8,gfs_raw,170,2.618202,3.456583,1.951261,22.941176,6.855059,10.588235


## Train-Fold 3% Missingness Audit


In [17]:
modeling_frame, candidate_categorical, candidate_numeric = _modeling_frame(result.features, config)
candidate_features = [*candidate_categorical, *candidate_numeric]
audit_specs = [
    (fold.name, fold.train_start_year, fold.train_end_year)
    for fold in V20_EXPANDING_FOLDS
] + [("test_refit_2021_2025", 2021, 2025)]

missingness_rows = []
years = pd.to_numeric(modeling_frame["year"], errors="coerce")
for fold_name, train_start, train_end in audit_specs:
    train = modeling_frame.loc[years.between(train_start, train_end)].copy()
    retained_categorical, retained_numeric = _fit_feature_columns(
        train,
        candidate_categorical,
        candidate_numeric,
        max_missing_fraction=config.effective_max_feature_missing_fraction,
    )
    retained = set(retained_categorical) | set(retained_numeric)
    for feature in candidate_features:
        numeric_feature = feature in candidate_numeric
        values = pd.to_numeric(train[feature], errors="coerce") if numeric_feature else train[feature]
        missingness_rows.append(
            {
                "fold": fold_name,
                "train_start_year": train_start,
                "train_end_year": train_end,
                "feature": feature,
                "kind": "numeric" if numeric_feature else "categorical",
                "missing_fraction": float(values.isna().mean()),
                "retained": feature in retained,
            }
        )

fold_feature_missingness = pd.DataFrame(missingness_rows)
retained_dropped_summary = (
    fold_feature_missingness.groupby(["fold", "retained"], as_index=False)
    .agg(feature_count=("feature", "nunique"), maximum_missing_fraction=("missing_fraction", "max"))
)
fold_feature_missingness.to_csv(config.resolved_output_dir() / f"{STATION_ID}_fold_feature_missingness.csv", index=False)
retained_dropped_summary, fold_feature_missingness.loc[~fold_feature_missingness["retained"]].sort_values(
    ["fold", "missing_fraction"], ascending=[True, False]
)


(                     fold  retained  feature_count  maximum_missing_fraction
 0  fold_2021_2022_to_2023     False             22                  0.931660
 1  fold_2021_2022_to_2023      True            183                  0.015342
 2  fold_2021_2023_to_2024     False             11                  0.929695
 3  fold_2021_2023_to_2024      True            194                  0.022202
 4  fold_2021_2024_to_2025     False             11                  0.930104
 5  fold_2021_2024_to_2025      True            194                  0.016609
 6       fold_2021_to_2022     False             22                  0.938202
 7       fold_2021_to_2022      True            183                  0.025281
 8    test_refit_2021_2025     False             11                  0.931454
 9    test_refit_2021_2025      True            194                  0.013267,
                        fold  train_start_year  train_end_year  \
 206  fold_2021_2022_to_2023              2021            2022   
 246  fol

## Expanded 11 AM Feature Coverage and Provider Count


In [18]:
new_feature_coverage = (
    result.features[V20_KDAL_1PM_TEMP_FEATURE_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)
provider_count_coverage = (
    result.features["v13sf_forecast_temp_1pm_provider_count"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("available_provider_count")
    .reset_index(name="row_count")
)
provider_count_coverage["row_pct"] = provider_count_coverage["row_count"] / len(result.features) * 100
new_feature_coverage.to_csv(config.resolved_output_dir() / f"{STATION_ID}_1pm_feature_coverage.csv", index=False)
new_feature_coverage, provider_count_coverage


(                                              feature  coverage_pct
 0                      v13sf_forecast_temp_1pm_mean_f         100.0
 1                    v13sf_forecast_temp_1pm_median_f         100.0
 2            v13sf_forecast_temp_1pm_minus_observed_f         100.0
 3                 v13sf_forecast_temp_1pm_abs_error_f         100.0
 4                v13sf_forecast_temp_1pm_warm_error_f         100.0
 5                v13sf_forecast_temp_1pm_cool_error_f         100.0
 6                    v13sf_forecast_temp_1pm_spread_f         100.0
 7              v13sf_forecast_temp_1pm_provider_count         100.0
 8   v13sf_forecast_temp_bias_remaining_warmup_inte...         100.0
 9          v13sf_observation_adjusted_provider_high_f         100.0
 10                  v13sf_forecast_warmup_after_1pm_f         100.0,
    available_provider_count  row_count   row_pct
 0                         2          9   0.45045
 1                         3       1989  99.54955)

## New-Feature Importance


In [19]:
new_feature_importance = result.feature_importance.loc[
    result.feature_importance["feature"].isin(V20_KDAL_1PM_TEMP_FEATURE_COLUMNS)
].sort_values(["method", "importance_mean_mae_f"], ascending=[True, False])
new_feature_importance


,method,param_key,feature,importance_mean_mae_f,importance_std_mae_f,n_repeats,train_start_year,train_end_year,test_year,train_rows,test_rows
3,catboost,trial_17,v13sf_forecast_warmup_after_1pm_f,0.103627,0.027826,10,2021,2025,2026,1809,170
24,catboost,trial_17,v13sf_forecast_temp_bias_remaining_warmup_inte...,0.018567,0.007666,10,2021,2025,2026,1809,170
79,catboost,trial_17,v13sf_forecast_temp_1pm_minus_observed_f,0.004906,0.002386,10,2021,2025,2026,1809,170
105,catboost,trial_17,v13sf_forecast_temp_1pm_warm_error_f,0.003724,0.001540,10,2021,2025,2026,1809,170
175,catboost,trial_17,v13sf_forecast_temp_1pm_abs_error_f,0.001491,0.001779,10,2021,2025,2026,1809,170
180,catboost,trial_17,v13sf_forecast_temp_1pm_cool_error_f,0.001419,0.000999,10,2021,2025,2026,1809,170
189,catboost,trial_17,v13sf_forecast_temp_1pm_mean_f,0.001359,0.000902,10,2021,2025,2026,1809,170
193,catboost,trial_17,v13sf_observation_adjusted_provider_high_f,0.001297,0.000432,10,2021,2025,2026,1809,170
341,catboost,trial_17,v13sf_forecast_temp_1pm_provider_count,0.000000,0.000000,10,2021,2025,2026,1809,170
399,catboost,trial_17,v13sf_forecast_temp_1pm_spread_f,-0.000032,0.001121,10,2021,2025,2026,1809,170


## 2026 Monthly Metrics


In [20]:
monthly_predictions = result.test_predictions.copy()
monthly_predictions["month"] = pd.to_datetime(monthly_predictions["contract_date"], errors="coerce").dt.month
monthly_metrics = (
    monthly_predictions.dropna(subset=["month", "error_f"])
    .groupby(["method", "month"], as_index=False)
    .agg(
        count=("error_f", "size"),
        mae_f=("absolute_error_f", "mean"),
        rmse_f=("error_f", lambda values: float(np.sqrt(np.mean(np.square(values))))),
        bias_f=("error_f", "mean"),
    )
)
monthly_metrics.to_csv(config.resolved_output_dir() / f"{STATION_ID}_2026_monthly_metrics.csv", index=False)
monthly_metrics


,method,month,count,mae_f,rmse_f,bias_f
0,catboost,1,31,0.868710,1.076473,0.061069
1,catboost,2,28,1.023909,1.320144,0.318520
2,catboost,3,30,1.157268,1.504499,0.645713
3,catboost,4,29,0.836199,1.075888,-0.270778
4,catboost,5,31,1.034983,1.427130,-0.056484
5,catboost,6,21,1.504640,1.854543,0.375086
6,gfs_raw,1,31,2.387726,3.136526,1.569948
7,gfs_raw,2,28,2.603194,3.157897,2.354177
8,gfs_raw,3,30,2.527869,4.046734,1.835206
9,gfs_raw,4,29,2.626329,3.323656,0.841344


## Performance by Warm/Cool 11 AM Forecast Delta


In [21]:
delta_by_date = result.features[[
    "contract_date",
    "v13sf_forecast_temp_1pm_minus_observed_f",
]].copy()
delta_predictions = result.test_predictions.merge(delta_by_date, on="contract_date", how="left")
delta_predictions["forecast_temp_delta_bucket"] = pd.cut(
    delta_predictions["v13sf_forecast_temp_1pm_minus_observed_f"],
    bins=[-np.inf, -2.0, -0.5, 0.5, 2.0, np.inf],
    labels=["cool_gt_2f", "cool_0.5_to_2f", "near_match", "warm_0.5_to_2f", "warm_gt_2f"],
)
warm_cool_metrics = (
    delta_predictions.dropna(subset=["forecast_temp_delta_bucket", "error_f"])
    .groupby(["method", "forecast_temp_delta_bucket"], observed=True, as_index=False)
    .agg(count=("error_f", "size"), mae_f=("absolute_error_f", "mean"), bias_f=("error_f", "mean"))
)
warm_cool_metrics.to_csv(config.resolved_output_dir() / f"{STATION_ID}_warm_cool_delta_metrics.csv", index=False)
warm_cool_metrics


,method,forecast_temp_delta_bucket,count,mae_f,bias_f
0,catboost,cool_gt_2f,46,1.034182,0.451167
1,catboost,cool_0.5_to_2f,53,0.964808,0.230960
2,catboost,near_match,33,1.003699,0.275899
3,catboost,warm_0.5_to_2f,23,1.241516,-0.278172
4,catboost,warm_gt_2f,15,1.191005,-0.482997
5,gfs_raw,cool_gt_2f,46,2.923059,2.738939
6,gfs_raw,cool_0.5_to_2f,53,1.826182,0.725302
7,gfs_raw,near_match,33,1.927716,-0.342528
8,gfs_raw,warm_0.5_to_2f,23,2.729520,-1.168412
9,gfs_raw,warm_gt_2f,15,5.830156,-0.454378


## Common-Date Comparison with V20 11 AM No Peak


In [22]:
baseline_path = (
    PROJECT_ROOT
    / "data"
    / "calibration"
    / "station_stacking_v20_kdal_no_peak"
    / f"{STATION_ID}_year_split_test_predictions.csv"
)
baseline_predictions = pd.read_csv(baseline_path)
baseline_predictions["contract_date"] = baseline_predictions["contract_date"].astype(str).str[:10]
fix_predictions = result.test_predictions.copy()
fix_predictions["contract_date"] = fix_predictions["contract_date"].astype(str).str[:10]
comparison = baseline_predictions.merge(
    fix_predictions,
    on=["contract_date", "method"],
    suffixes=("_baseline", "_fix"),
)
comparison["baseline_abs_error_f"] = (
    pd.to_numeric(comparison["actual_high_f_baseline"], errors="coerce")
    - pd.to_numeric(comparison["predicted_high_f_baseline"], errors="coerce")
).abs()
comparison["fix_abs_error_f"] = (
    pd.to_numeric(comparison["actual_high_f_fix"], errors="coerce")
    - pd.to_numeric(comparison["predicted_high_f_fix"], errors="coerce")
).abs()
common_date_comparison = (
    comparison.groupby("method", as_index=False)
    .agg(
        common_date_count=("contract_date", "size"),
        baseline_mae_f=("baseline_abs_error_f", "mean"),
        fix_mae_f=("fix_abs_error_f", "mean"),
        fix_better_days=("fix_abs_error_f", lambda values: int((values < comparison.loc[values.index, "baseline_abs_error_f"]).sum())),
        baseline_better_days=("fix_abs_error_f", lambda values: int((values > comparison.loc[values.index, "baseline_abs_error_f"]).sum())),
    )
)
common_date_comparison["delta_mae_f"] = common_date_comparison["fix_mae_f"] - common_date_comparison["baseline_mae_f"]
common_date_comparison.to_csv(config.resolved_output_dir() / f"{STATION_ID}_1pm_vs_11am_common_date_comparison.csv", index=False)
common_date_comparison.sort_values("delta_mae_f")


,method,common_date_count,baseline_mae_f,fix_mae_f,fix_better_days,baseline_better_days,delta_mae_f
0,catboost,170,1.347432,1.048525,103,64,-0.298907
3,lightgbm,170,1.341638,1.064216,98,68,-0.277423
7,ridge_stack,170,1.298176,1.068964,98,72,-0.229212
2,hrrr_raw,170,4.927253,4.742768,97,73,-0.184485
8,xgboost,170,1.325696,1.143512,88,77,-0.182184
6,provider_median,170,2.428105,2.371618,56,44,-0.056488
5,provider_mean,170,2.583543,2.532665,102,68,-0.050878
4,nbm_raw,170,2.045105,2.035190,63,59,-0.009914
1,gfs_raw,170,2.550131,2.618202,47,63,0.068071
